In [51]:
import pandas as pd
from collections import Counter

# -----------------------------
# 1. Load CSV
# -----------------------------
df = pd.read_csv(
    "data/isolation_sources_bacdive_2026-02-11.csv"
)

# -----------------------------
# 2. Extract genus from Species
# -----------------------------
df["Genus"] = df["Species"].astype(str).str.split().str[0]

# -----------------------------
# 3. Keep only rows with Cat1 info
# -----------------------------
df = df[df["Category 1"].notna()].copy()

# -----------------------------
# 4. Collapse to strain-level Cat1 sets (each ID = one strain)
# -----------------------------
strain_cat1 = (
    df.groupby(["Genus", "ID"])["Category 1"]
      .apply(lambda x: set(x.dropna()))
      .reset_index()
)

# -----------------------------
# 5. Compute genus-level proportions (minimal output)
# -----------------------------
HOST_TAGS = {"#Host", "#Host Body-Site", "#Host Body Product", "#Infection"}

rows = []

for genus, sub in strain_cat1.groupby("Genus"):
    n_strains = len(sub)  # denominator (total strains in this genus)

    counter = Counter()
    for tagset in sub["Category 1"]:
        for tag in tagset:
            counter[tag] += 1

    # numerators (counts of strains with each signal)
    n_env = counter.get("#Environmental", 0)
    n_eng = counter.get("#Engineered", 0)
    n_host = sum(counter.get(t, 0) for t in HOST_TAGS)

    # proportions
    p_env = n_env / n_strains if n_strains else None
    p_eng = n_eng / n_strains if n_strains else None
    p_host = n_host / n_strains if n_strains else None

    # -------------------------
    # 6. Tier label (host-sensitive thresholds you chose)
    # -------------------------
    if p_env is None:
        label = "no_data"
    elif p_env >= 0.80 and p_host <= 0.10:
        label = "highly_likely_environmental"
    elif p_env >= 0.60:
        label = "could_be_environmental"
    elif p_host >= 0.60 and p_env <= 0.40:
        label = "highly_likely_host_associated"
    elif p_host >= 0.40:
        label = "likely_host_associated"
    else:
        label = "mixed_or_uncertain"

    rows.append({
        "Genus": genus,
        "n_strains": n_strains,  # denominator

        # numerators
        "n_env": n_env,
        "n_host": n_host,
        "n_eng": n_eng,

        # proportions
        "p_env": p_env,
        "p_host": p_host,
        "p_eng": p_eng,

        "label": label,
    })

genus_df = pd.DataFrame(rows)

# -----------------------------
# 7. Clean + sort
# -----------------------------
genus_df = genus_df.dropna(subset=["Genus"])
genus_df = genus_df[genus_df["Genus"].astype(str).str.strip() != ""]
genus_df = genus_df.sort_values("p_env", ascending=False).reset_index(drop=True)

# -----------------------------
# 8. Save final table
# -----------------------------
genus_df.to_csv(
    "data/bacdive_genus_ecology_labels.csv",
    index=False
)


In [52]:
genus_df

,Genus,n_strains,n_env,n_host,n_eng,p_env,p_host,p_eng,label
0,Acidaminobacter,1,1,0,0,1.0,0.0,0.0,highly_likely_environmental
1,Zooshikella,2,2,0,0,1.0,0.0,0.0,highly_likely_environmental
2,Abditibacterium,1,1,0,0,1.0,0.0,0.0,highly_likely_environmental
3,Acetohalobium,1,1,0,0,1.0,0.0,0.0,highly_likely_environmental
4,Abyssibacter,1,1,0,0,1.0,0.0,0.0,highly_likely_environmental
...,...,...,...,...,...,...,...,...,...
3050,Planctomyces,1,0,1,0,0.0,1.0,0.0,highly_likely_host_associated
3051,Planctomicrobium,1,0,0,0,0.0,0.0,0.0,mixed_or_uncertain
3052,Pisciglobus,1,0,0,1,0.0,0.0,1.0,mixed_or_uncertain
3053,Piscicoccus,1,0,1,0,0.0,1.0,0.0,highly_likely_host_associated


In [50]:
genus_df['label'].value_counts()

label
highly_likely_host_associated    918
highly_likely_environmental      876
mixed_or_uncertain               822
could_be_environmental           230
likely_host_associated           209
Name: count, dtype: int64

---

In [1]:
import pandas as pd

genus_df = pd.read_csv(
    "data/bacdive_genus_ecology_labels.csv"
)

In [2]:
genus_df

,Genus,n_strains,n_env,n_host,n_eng,p_env,p_host,p_eng,label
0,Acidaminobacter,1,1,0,0,1.0,0.0,0.0,highly_likely_environmental
1,Zooshikella,2,2,0,0,1.0,0.0,0.0,highly_likely_environmental
2,Abditibacterium,1,1,0,0,1.0,0.0,0.0,highly_likely_environmental
3,Acetohalobium,1,1,0,0,1.0,0.0,0.0,highly_likely_environmental
4,Abyssibacter,1,1,0,0,1.0,0.0,0.0,highly_likely_environmental
...,...,...,...,...,...,...,...,...,...
3050,Planctomyces,1,0,1,0,0.0,1.0,0.0,highly_likely_host_associated
3051,Planctomicrobium,1,0,0,0,0.0,0.0,0.0,mixed_or_uncertain
3052,Pisciglobus,1,0,0,1,0.0,0.0,1.0,mixed_or_uncertain
3053,Piscicoccus,1,0,1,0,0.0,1.0,0.0,highly_likely_host_associated
